In [ ]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd

import parcels
from parcels import StatusCode


In [ ]:
refine_lon_fac = 5
refine_lat_fac = 5

YEAR = 2020
integration_days = 5
integration_direction = -1

max_sweep_dates = None

fname = "../data/hourly_glorys/cabo_verde_TUV_20_25_hourly_glorys_landmasked.nc"
fname_edge = "../data/hourly_glorys/cabo_verde_TUV_20_25_hourly_glorys_landmasked_zero_edges.nc"
ftle_output_dir = "../data/ftle_calc_and_plots_tests"


In [ ]:
reference_times = [t.strftime("%Y-%m-%d")
                    for t in pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq=f"{integration_days}D")]
if max_sweep_dates is not None:
    reference_times = reference_times[:max_sweep_dates]
print(f"{len(reference_times)} reference dates for the {integration_days}-day FTLE sweep in {YEAR}")


In [ ]:
ds = xr.open_dataset(fname)


In [ ]:
# this file is identical every run (doesn't depend on YEAR/integration_days), so
# only build it once; written via temp file + atomic rename so a kill mid-write
# (e.g. OOM) can't leave behind a corrupt-but-"finished" file
tmp_edge = fname_edge + ".tmp"

if Path(fname_edge).exists():
    print(f"{fname_edge} already exists, skipping edge-zeroing rebuild")
else:
    if Path(tmp_edge).exists():
        raise FileExistsError(
            f"{tmp_edge} exists from a previous interrupted run -- delete it and re-run."
        )

    edge_x = xr.DataArray(np.zeros(ds.sizes["x"], dtype=bool), dims="x")
    edge_x[[0, -1]] = True
    edge_y = xr.DataArray(np.zeros(ds.sizes["y"], dtype=bool), dims="y")
    edge_y[[0, -1]] = True
    is_edge = edge_x | edge_y

    for var in ["sozocrtx", "somecrty"]:
        dims = ds[var].dims
        ds[var] = xr.where(is_edge, 0.0, ds[var], keep_attrs=True).transpose(*dims).astype(ds[var].dtype)

    ds.to_netcdf(tmp_edge)
    Path(tmp_edge).rename(fname_edge)


In [ ]:
filenames_uv = {
    "U": {"lon": fname_edge, "lat": fname_edge, "data": fname_edge},
    "V": {"lon": fname_edge, "lat": fname_edge, "data": fname_edge},
}
variables_uv = {"U": "sozocrtx", "V": "somecrty"}
dimensions_uv = {"lon": "lon_f", "lat": "lat_f", "time": "time_counter"}

fieldset = parcels.FieldSet.from_nemo(filenames_uv, variables_uv, dimensions_uv, allow_time_extrapolation=True)
fieldset.computeTimeChunk()


In [ ]:
lon_min, lon_max = float(fieldset.U.grid.lon.min()), float(fieldset.U.grid.lon.max())
lat_min, lat_max = float(fieldset.U.grid.lat.min()), float(fieldset.U.grid.lat.max())

particle_lon = xr.DataArray(np.linspace(lon_min, lon_max, refine_lon_fac * ds.sizes["x"] - 1), name="plon", dims="plon")
particle_lat = xr.DataArray(np.linspace(lat_min, lat_max, refine_lat_fac * ds.sizes["y"] - 1), name="plat", dims="plat")
particle_lon, particle_lat = xr.broadcast(particle_lon, particle_lat)
print(f"{particle_lon.size} particles per reference date")


In [ ]:
def compute_ftle(reference_time):
    pset = parcels.ParticleSet.from_list(
        fieldset=fieldset, pclass=parcels.JITParticle,
        lon=particle_lon.stack(pid=["plon", "plat"]).load().data,
        lat=particle_lat.stack(pid=["plon", "plat"]).load().data,
        time=np.datetime64(reference_time),
    )
    # parcels particle ids keep incrementing across every ParticleSet.from_list()
    # call in this process instead of resetting to 0 each time, so capture each
    # particle's own starting id here and index relative to it below
    id_offset = int(pset.id.min())

    def delete_particle(particle, fieldset, time):
        if particle.state == StatusCode.ErrorOutOfBounds:
            particle.delete()

    kernels = pset.Kernel(parcels.AdvectionRK4) + pset.Kernel(delete_particle)
    pset.execute(kernels, runtime=timedelta(days=integration_days),
                 dt=timedelta(minutes=int(integration_direction * 5)))

    total = particle_lon.size
    final_lon_flat = np.full(total, np.nan)
    final_lat_flat = np.full(total, np.nan)
    final_lon_flat[pset.id - id_offset] = pset.lon
    final_lat_flat[pset.id - id_offset] = pset.lat

    final_lon = xr.DataArray(final_lon_flat.reshape(particle_lon.shape),
                              dims=particle_lon.dims, coords=particle_lon.coords, name="final_lon")
    final_lat = xr.DataArray(final_lat_flat.reshape(particle_lat.shape),
                              dims=particle_lat.dims, coords=particle_lat.coords, name="final_lat")

    dx_dlon = (final_lon.shift(plon=-1) - final_lon.shift(plon=1)) * 111e3 * np.cos(np.deg2rad(final_lat))
    dx_dlat = (final_lon.shift(plat=-1) - final_lon.shift(plat=1)) * 111e3 * np.cos(np.deg2rad(final_lat))
    dy_dlon = (final_lat.shift(plon=-1) - final_lat.shift(plon=1)) * 111e3
    dy_dlat = (final_lat.shift(plat=-1) - final_lat.shift(plat=1)) * 111e3

    dlon = (particle_lon.shift(plon=-1) - particle_lon.shift(plon=1)) * 111e3 * np.cos(np.deg2rad(particle_lat))
    dlat = (particle_lat.shift(plat=-1) - particle_lat.shift(plat=1)) * 111e3

    f = np.array([
        [(dx_dlon / dlon).stack(pid=["plon", "plat"]), (dx_dlat / dlat).stack(pid=["plon", "plat"])],
        [(dy_dlon / dlon).stack(pid=["plon", "plat"]), (dy_dlat / dlat).stack(pid=["plon", "plat"])],
    ])

    c = np.einsum("kil,kjl->ijl", f, f)
    eigenvalues, _ = np.linalg.eigh(np.transpose(c, (2, 0, 1)))

    t_seconds = integration_days * 24 * 60 * 60
    max_eigenvalues = (np.max(eigenvalues, axis=1) + particle_lon.stack(pid=["plon", "plat"]) * 0).rename("lambda_plus").unstack()
    max_eigenvalues = max_eigenvalues.assign_coords(
        plon=particle_lon.isel(plat=0, drop=True), plat=particle_lat.isel(plon=0, drop=True),
    )

    ftle = (1 / (2 * t_seconds)) * np.log(np.maximum(max_eigenvalues, 1e-10))
    return ftle


In [ ]:
Path(ftle_output_dir).mkdir(parents=True, exist_ok=True)

ftle_by_date = {}
for reference_time in reference_times:
    out_path = Path(f"{ftle_output_dir}/010_FTLE_{integration_days}d_{reference_time}.nc")
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")

    if out_path.exists():
        print(f"{out_path.name} already exists, loading from disk (skipping recompute)")
        ftle_by_date[reference_time] = xr.open_dataarray(out_path)
        continue

    if tmp_path.exists():
        print(f"Removing incomplete {tmp_path.name} from a previous interrupted run")
        tmp_path.unlink()

    print(f"Computing {integration_days}-day FTLE for {reference_time} ...")
    ftle = compute_ftle(reference_time)
    ftle.drop_encoding().to_netcdf(tmp_path)
    tmp_path.rename(out_path)
    ftle_by_date[reference_time] = ftle
